# Multiple Object Video Detection
## Ensemble

Downloading scripts and chepoints from google drive

In [1]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=True)
BASE = Path("/content/drive/MyDrive/ai-hw2")
CODE, RUNS, REPORT = BASE / "code", BASE / "runs", BASE / "eda_report"
for p in (CODE, RUNS, REPORT):
    p.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))

needed = ["scripts/eda.py", "scripts/fr_cnn.py", "scripts/run_experiments.py", "scripts/analyze_model.py", "scripts/detection_errors.py", "scripts/anchor_analysis.py", "scripts/retinanet.py", "scripts/run_retina_net.py", "scripts/analyze_retina_net.py", "scripts/make_submission.py"]
missing = [f for f in needed if not (CODE / f).exists()]
assert not missing, f"copy these files to {CODE}: {missing}"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime type")
print("CPU cores:", os.cpu_count())

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_ACCESS_TOKEN").strip()

Mounted at /content/drive
torch 2.14.1+cu130 | GPU: Tesla T4
CPU cores: 2


Downloading and extracting the data

In [2]:
!kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026
!unzip ./multiple-object-video-detection-hw-2-ucu-ai-2026.zip -d ./object-detection-data

Streaming output truncated to the last 5000 lines.
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000229.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000230.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000231.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000232.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000233.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000234.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000235.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000236.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000237.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000238.jpg  
  inflating: ./object-detection-data/val/images/uav0000117_02622_v/0000239.jpg  
  inflating: ./object-detection-data/val/images/uav0000117

### Preparing the dataset

In [3]:
import scripts.eda as eda
DATA_ROOT = Path("./object-detection-data")
PREPARED = Path("/content/prepared")
STEP = 3
if not (PREPARED / "data.yaml").exists():
    eda.rebuild_dataset(DATA_ROOT, PREPARED, STEP)
print("prepared dataset:", PREPARED)

OK: every clip belongs to exactly one split
Split summary
       clips  frames  objects  objs_per_frame
split                                        
train     55   22481  1013421            45.1
val        8    4566   206227            45.2
Class mix per split [%]
       person  bicycle    car   van  truck  tricycle   bus  motor
split                                                            
train   30.75     3.76  43.94  4.30   2.43      4.02  0.63  10.18
val     32.67     3.98  44.55  4.96   3.49      2.96  1.72   5.66
Subsampling step 3: 27047 -> 9041 frames

wrote 9041 frames to /content/prepared
prepared dataset: /content/prepared


As FasterCNN performed better than RetinaNet we will mostly focus on it to improve our score.

In [ ]:
tv = D.train_vs_val(RUN_DIR)
if tv is not None:
    display(tv.round(3))

,train subset,validation
mAP,0.632,0.193
AP50,0.948,0.413
AP75,0.713,0.155
AP_small,0.492,0.076
AP_medium,0.638,0.183
AP_large,0.730,0.297


The output of the cell above (for some reason it refuses to be displayed when I download the colab):
	train subset	validation
mAP	0.632	0.193
AP50	0.948	0.413
AP75	0.713	0.155
AP_small	0.492	0.076
AP_medium	0.638	0.183
AP_large	0.730	0.297

Those results show that model has already overfit very much so just training it further won't work.

### Choosing models for ensemble

We take all our best models as candidates for our ensemble.
Below we collect their detections on validation split and compare fusion performance. To fuse the prediction we take bounding boxes of all the models for one image. Then we evaluate fusion performance with different IoU threshold.

In [ ]:
import scripts.ensemble as E
import scripts.fr_cnn as L

best_run = RUNS / "final4" / "final4"
cfg_b = L.load_config(best_run)
val_ds, coco_gt = L.build_val(cfg_b)
specs = [
  {"run": RUNS/"final4"/"final4", "kind": "faster", "test_imgsz": 1280},
  {"run": RUNS/"final4"/"final4", "kind": "faster", "flips": (True,)},
  {"run": RUNS/"final4"/"final4", "kind": "faster", "test_imgsz": 1920, "flips": (True,)},
  {"run": RUNS/"retina_tune_1536"/"initial_1536", "kind": "retina", "test_imgsz": 1920, "weight": 0.7},
  {"run": RUNS/"retina_tune_1536"/"anchors_only_1536", "kind": "retina", "weight": 0.5},
]
specs = [s for s in specs if (Path(s["run"]) / "last.pt").exists()]
named = E.collect_detections(specs, val_ds, L.get_device(), RUNS / "det_cache", "val")
if named:
    display(E.compare_fusion(named, coco_gt, iou_thrs=(0.5, 0.6, 0.7)).round(4))

,mAP,AP50,AP75,AP_small
final4,0.1737,0.3701,0.1436,0.0627
final4_flip,0.2107,0.4639,0.1592,0.0778
initial_1536,0.1763,0.3661,0.1457,0.0487
WBF iou_thr=0.5,0.2261,0.4749,0.1849,0.0892
WBF iou_thr=0.6,0.2285,0.4759,0.1876,0.0897
WBF iou_thr=0.7,0.2290,0.4724,0.1899,0.0887


The output of the cell above also vanished when I download it:

mAP	AP50	AP75	AP_small
final4	0.1737	0.3701	0.1436	0.0627
final4_flip	0.2107	0.4639	0.1592	0.0778
initial_1536	0.1763	0.3661	0.1457	0.0487
WBF iou_thr=0.5	0.2261	0.4749	0.1849	0.0892
WBF iou_thr=0.6	0.2285	0.4759	0.1876	0.0897
WBF iou_thr=0.7	0.2290	0.4724	0.1899	0.0887


We can see that the best performance was achieved while fusing with the threshold = 0.7.

In [ ]:
named = E.collect_detections(specs, val_ds, L.get_device(), RUNS / "det_cache", "val")

labels = []
for s in specs:
    for flip in s.get("flips", (False,)):
        labels.append(Path(s["run"]).name + ("_flip" if flip else "")
                      + (f"_{s['test_imgsz']}" if s.get("test_imgsz") else ""))
assert len(labels) == len(named), (labels, [n for n, _, _ in named])
named = [(lab, d, w) for lab, (_, d, w) in zip(labels, named)]
print(labels)

by_name = {n: (n, d, w) for n, d, w in named}

Above we saw that this approach is successful and now we need to find the best combination of model. Below we took the 3 FasterCNN model as a baseline and checked if adding RetinaNet will be successfull. 

In [ ]:
from scripts.fr_cnn import coco_evaluate

names = [n for n, _, _ in named]
print("members:", names)
assert len(set(names)) == len(names), "duplicate member names: relabel them first (see my previous message)"
by_name = {n: (n, d, w) for n, d, w in named}

base = names[:3]
cur = list(base)

def score(sel, thr=0.7):
    m = coco_evaluate(coco_gt, E.fuse_detections([by_name[n] for n in sel], thr))
    return m["mAP"], m["AP75"], m["AP_small"]

best = score(cur)
print("start", base, [round(x, 4) for x in best])
for n in [n for n in names if n not in base]:
    s = score(cur + [n]); print(n, [round(x, 4) for x in s])
    if s[0] > best[0] + 0.002:
        cur.append(n); best = s
print("selected:", cur, [round(x, 4) for x in best])

members: ['final4_1280', 'final4_flip', 'final4_flip_1920', 'initial_1536_1920']
start ['final4_1280', 'final4_flip', 'final4_flip_1920'] [0.2237, np.float64(0.1831), np.float64(0.0878)]
initial_1536_1920 [0.229, np.float64(0.1899), np.float64(0.0887)]
selected: ['final4_1280', 'final4_flip', 'final4_flip_1920', 'initial_1536_1920'] [0.229, np.float64(0.1899), np.float64(0.0887)]


In [ ]:
import scripts.ensemble as E

sub = E.ensemble_submission(
    specs,
    DATA_ROOT,
    RUNS / "submissions" / "submission_ensemble.csv",
    L.get_device(),
    iou_thr=0.7,
    cache_dir=RUNS / "det_cache")

detecting with final4 ...
detecting with final4_flip ...
detecting with final4_flip ...
detecting with initial_1536 ...
wrote /content/drive/MyDrive/ai-hw2/runs/submissions/submission_ensemble.csv: {'images': 3811, 'empty': 0, 'detections': 1903786, 'mean_per_image': 499.55024927840464, 'max_per_image': 500}


In [ ]:
from scripts.fr_cnn import coco_evaluate
rows = []
for w_r in (0.5, 1.0):
    nm = [(n, d, w_r if n.startswith("initial") else w) for n, d, w in named]
    for thr in (0.7, 0.8):
        m = coco_evaluate(coco_gt, E.fuse_detections(nm, thr))
        rows.append({"retina_weight": w_r, "iou_thr": thr, **{k: m[k] for k in ("mAP", "AP50", "AP75", "AP_small")}})
import pandas as pd
pd.DataFrame(rows).round(4)

,retina_weight,iou_thr,mAP,AP50,AP75,AP_small
0,0.5,0.7,0.2290,0.4751,0.1881,0.0976
1,0.5,0.8,0.2258,0.4612,0.1893,0.0947
2,1.0,0.7,0.2268,0.4677,0.1882,0.0968
3,1.0,0.8,0.2237,0.4530,0.1896,0.0945


In [ ]:
import pandas as pd
pd.DataFrame(rows).round(4)

,retina_weight,iou_thr,mAP,AP50,AP75,AP_small
0,0.5,0.7,0.2079,0.4289,0.1756,0.0852
1,0.5,0.8,0.2068,0.4227,0.1759,0.0841
2,1.0,0.7,0.2046,0.4183,0.1754,0.0834
3,1.0,0.8,0.2026,0.4092,0.1760,0.0816
